# Desmos-Art-Generator

This notebook is for you to learn.

The pipeline is

image -> gray_image -> blur_image -> edge detection -> potrace to draw vector -> turn vector into desmos graph

In [ ]:
%%capture
!apt-get install build-essential python3-dev libagg-dev libpotrace-dev pkg-config
!pip install pypotrace

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import potrace
from skimage import data, color, feature

In [ ]:
def ensure_grayscale(image):
    if len(image.shape) == 3:
        if image.shape[2] in [3, 4]:
            return color.rgb2gray(image)
    return image

def apply_canny(image_gray, sigma=1, low_threshold=None, high_threshold=None):
    edges = feature.canny(image_gray, sigma, low_threshold, high_threshold)
    return edges.astype(np.uint8)

# --- DESMOS FORMATTERS ---
def get_line_latex(p1, p2):
    # Standard Parametric Line: (1-t)P1 + tP2
    return f"((1-t)*{p1[0]}+t*{p2[0]}, (1-t)*{-p1[1]}+t*{-p2[1]})"

def get_bezier_latex(p1, c1, c2, p2):
    # Cubic Bezier Parametric Equation
    x = f"(1-t)^3*{p1[0]} + 3t(1-t)^2*{c1[0]} + 3t^2(1-t)*{c2[0]} + t^3*{p2[0]}"
    y = f"(1-t)^3*{-p1[1]} + 3t(1-t)^2*{-c1[1]} + 3t^2(1-t)*{-c2[1]} + t^3*{-p2[1]}"
    return f"({x}, {y})"

def generate_equations(path):
    eqs = []
    for curve in path:
        # The starting point of the path
        start_point = curve.start_point

        for segment in curve:
            match segment:
                case potrace.CornerSegment(c=p1, end_point=p2):
                    # p0 is start, p1 is corner, p2 is end
                    p0 = start_point
                    eqs.append(get_line_latex(p0, p1))
                    eqs.append(get_line_latex(p1, p2))
                    start_point = p2

                case potrace.BezierSegment(c1=p1, c2=p2, end_point=p3):
                    # p0 is start, p1/p2 are controls, p3 is end
                    p0 = start_point
                    eqs.append(get_bezier_latex(p0, p1, p2, p3))
                    start_point = p3
    return eqs

## Edge detection

This is what canny edge detection does to a gray scale image

In [ ]:
image = data.astronaut()

image_gray = ensure_grayscale(image)

result = apply_canny(image_gray)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image_gray, cmap='gray')
axes[0].set_title("Original (Grayscale)")
axes[0].axis('off')

axes[1].imshow(result, cmap='gray')
axes[1].set_title("Canny Edge Detection")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Full pipeline

In [ ]:
def image_to_desmos(image, sigma=3, low_threshold=None, high_threshold=None, turd_size=5):
    # 1. ENSURE GRAYSCALE
    image_gray = ensure_grayscale(image)

    # 2. APPLY CANNY EDGE DETECTION
    binary_map = apply_canny(image_gray, sigma, low_threshold, high_threshold)

    # 2. CONVERT TO INTEGER
    binary_map = binary_map.astype(np.uint8)

    # 3. TRACING BITMAP INTO PATH
    bmp = potrace.Bitmap(binary_map)
    path = bmp.trace(turdsize=turd_size)

    # 4. GENERATE EQUATIONS
    return generate_equations(path)

In [ ]:
equations = image_to_desmos(image)

print("number of equations: " + str(len(equations)))
if len(equations) > 1000:
  print("⚠️ Be careful when plotting on desmos. Your browser might crashed if not enough RAM")

In [ ]:
for e in equations:
  print(e)

Thanks [Gemini](https://gemini.google.com/), [Google Colab](https://colab.research.google.com/)